In [1]:
import akshare as ak
import pandas as pd
from pandas.tseries.offsets import BDay
import numpy as np
from datetime import datetime, timedelta

In [2]:
### Get Stock List
directory_path = '0-CN_A_Stocks_Data/'
stock_list_path = 'stock_list.csv'
stock_financial_sheet_dir = '0-Financial_Profit_Sheets/'
stock_list = pd.read_csv(directory_path + stock_list_path)
stock_list.head(3)

,代码,名称,最新价,涨跌额,涨跌幅,买入,卖出,昨收,今开,最高,最低,成交量,成交额,时间戳
0,bj430017,星昊医药,23.72,0.89,3.898,23.70,23.72,22.83,22.48,24.50,22.31,5488047.0,128380318.0,10:08:36
1,bj430047,诺思兰德,26.03,0.78,3.089,26.03,26.07,25.25,24.82,26.49,24.52,6509503.0,167507174.0,10:08:36
2,bj430090,同辉信息,9.14,-0.12,-1.296,9.14,9.15,9.26,9.25,9.30,9.11,2412254.0,22178998.0,10:08:36


In [3]:
def store_profit_sheet(df):
    """
    Store profit sheet locally.
    Note: By current logic, the old file will be overwritten. Consider appending new data (but probably unecessary).
    
    Parameters
    ----------
    df: pd.dataframe
        A dataframe holding the stock's profit sheet.

    Returns
    -------
    NA
    """
    file_name = stock_financial_sheet_dir + df['SECURITY_CODE'][0] + '_'+ df['SECURITY_NAME_ABBR'][0] + "_profit_sheet.csv"
    df.to_csv(file_name, index=False)

In [4]:
def get_stock_value(stock_list):
    """
    Gets and returns stock market value info.
    Note: Same code as Stocks_3GoldenCrosses_v1.1 MAIN CODE BLOCK 2.
          To take out and put into the data retrieval program.

    Parameters
    stocks: pd.dataframe
        A dataframe of stocks basic info.

    Returns
    -------
    stock_market_info: pd.dataframe
        A dataframe of market info.
    """
    stock_info = []
    for code in stock_list['代码'].head(3): ## FOR TESTING
        try:
            # Get info for individual stock
            stock_info_temp = ak.stock_individual_info_em(
                symbol = code[2:], # Unlike 三金叉, Exchange info IS NOT removed in target_stocks
                timeout = None                #Default value
            )
            # Append data
            stock_info.append(
                dict(zip(list(stock_info_temp['item']), list(stock_info_temp['value'])))
            )
        except Exception as e:
            print(f"股票 {code} 市值获取失败: {e}")
            continue

    return pd.DataFrame(stock_info)

#def get_stock_value(code):
#  """ Read a single stock market value file. Assumption: local datafile is up to date."""

In [7]:
### Main Code Block 1 - Get Data.
### For each stock, 1. acquire the financial info, 2. save it as a file, 
### 3. obtain 净利润 & other values, append to dataframe, continue.
profit_info = []
stock_market_info_df = get_stock_value(stock_list)  ## TO REMOVE, when local storage is complete.
stock_market_info_df = stock_market_info_df[['股票代码', '总股本', '流通股', '总市值', '流通市值', '行业']]


for code in stock_list['代码'].head(3): ## FOR TESTING
    try:
        # Get profit sheet for this stock
        # stock_profit_sheet_by_report_em_df = ak.stock_profit_sheet_by_report_em(symbol=code) ### 按报告期读取
        report_df = ak.stock_profit_sheet_by_quarterly_em(symbol=code)  ### 按季度读取
        # Store the data as a file.
        store_profit_sheet(report_df)
        
        # Put together the needed info, and append to list.
        if len(report_df) == 0:
            print('Stock ', code, 'has no available profit charts.')
            continue        
        temp_info = {'股票代码': report_df['SECURITY_CODE'].values[0],
                     '股票名称': report_df['SECURITY_NAME_ABBR'].values[0],
                     '最近报告期': report_df['REPORT_DATE'].values[0]}

        ## 添加 股票市值、市盈率、股价、行业 信息
        ### TO UPDATE WHEN LOCAL STORAGE IS COMPLETE
        single_stock_info = ak.stock_a_indicator_lg(code[2:]) #乐咕乐股-A 股个股指标: 市盈率, 市净率, 股息率 ##可以利用这个数据在别的地方用

        temp_info.update({
                     '最新价': stock_list[stock_list['代码']==code]['最新价'].values[0],
                     '市盈率': single_stock_info.iloc[-1]['pe'],
                     '总股本': stock_market_info_df[stock_market_info_df['股票代码']==code[2:]]['总股本'].values[0],
                     '流通股': stock_market_info_df[stock_market_info_df['股票代码']==code[2:]]['流通股'].values[0],
                     '总市值': stock_market_info_df[stock_market_info_df['股票代码']==code[2:]]['总市值'].values[0],
                     '流通市值': stock_market_info_df[stock_market_info_df['股票代码']==code[2:]]['流通市值'].values[0],
                     '行业': stock_market_info_df[stock_market_info_df['股票代码']==code[2:]]['行业'].values[0]})

        for i in range(min(3, len(report_df))):
            key_1 = report_df['REPORT_DATE_NAME'][i] + '净利润'
            val_1 = report_df['NETPROFIT'][i]
            key_2 = report_df['REPORT_DATE_NAME'][i] + '净利润环比上季度（%）'
            val_2 = report_df['NETPROFIT_QOQ'][i]
            temp_info.update({key_1: val_1, key_2: val_2})
        profit_info.append(temp_info)

    except Exception as e:
        print(f"股票 {code} 处理失败: {e}")
        continue

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/7 [00:00<?, ?it/s]

In [8]:
### Main Code Block 2 - Sort data and print to file.
profit_info_df = pd.DataFrame(profit_info)
newest_reporting_period = "2025一季度" + "净利润环比上季度（%）" ### MANUAL CHANGE THIS EACH SEASON ###
profit_info_df = profit_info_df.sort_values(by=newest_reporting_period, ascending=False)

profit_info_df.to_csv("profit_sheet_info.csv", index=False)

In [9]:
profit_info_df

,股票代码,股票名称,最近报告期,最新价,市盈率,总股本,流通股,总市值,流通市值,行业,2025一季度净利润,2025一季度净利润环比上季度（%）,2024四季度净利润,2024四季度净利润环比上季度（%）,2024三季度净利润,2024三季度净利润环比上季度（%）
0,430017,星昊医药,2025-03-31 00:00:00,23.72,33.6842,122288200.0,121417900.0,2.912905e+09,2.892174e+09,化学制药,25031096.91,508.646436,4112584.16,-77.541701,18312090.91,-47.687847
2,430090,同辉信息,2025-03-31 00:00:00,9.14,NaN,199333546.0,135347400.0,1.833869e+09,1.245196e+09,软件开发,-5974458.18,86.105594,-42999018.38,-121.559147,-19407467.02,-98.829609
1,430047,诺思兰德,2025-03-31 00:00:00,26.03,NaN,274271974.0,180106427.0,7.024105e+09,4.612526e+09,生物制品,-6783601.29,28.074932,-9431483.95,-50.194097,-6279530.40,50.653415
